# GOCART dust emission (Ginoux et al., 2001) in a MUSICA box model

The physics is in `quacs/dust/ginoux/ginoux.py`. The MUSICA box model is in `quacs/dust/ginoux/box_model.py`. This notebook sets the inputs, runs the box model, and plots the results.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from quacs.dust.ginoux import (
    LAND,
    RADIUS_DEFAULT,
    RHOP_DEFAULT,
    SP_DEFAULT,
    dust_emission,
    run_box_model,
    threshold_velocity,
)

## 1. Inputs

In [ ]:
w10m = 12.0  # 10-m wind speed [m s-1]
gwet = 0.1  # gravimetric soil moisture [-]
S = 0.5  # erodibility [-]
rhoa = 1.2  # near-surface air density [kg m-3]
dz = 50.0  # lowest-layer thickness [m]
dt = 120.0  # time step [s]
n_steps = 30

## 2. Run the MUSICA box model

In [ ]:
out = run_box_model(w10m=w10m, gwet=gwet, S=S, rhoa=rhoa, dz=dz, dt=dt, n_steps=n_steps)

expected = out["rates"].sum() * out["times"]
assert np.allclose(out["concs"].sum(axis=1), expected, rtol=1e-10)
print(f"bin-mean threshold u_t = {out['u_t']:.3f} m/s")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for i, r in enumerate(RADIUS_DEFAULT):
    ax.plot(out["times"] / 60, out["concs"][:, i], label=f"bin {i + 1} (r = {r * 1e6:.2f} um)")
ax.set_xlabel("Time (min)")
ax.set_ylabel("Dust mass (kg m$^{-3}$)")
ax.set_title("Dust mass for each bin")
ax.legend()
plt.tight_layout()

## 3. Emission flux as a function of wind speed and soil moisture

In [ ]:
u_t_bins = threshold_velocity(RADIUS_DEFAULT, RHOP_DEFAULT, np.array([[rhoa]]))
u_t = np.array([[u_t_bins[0, 0, :].mean()]])
winds = np.linspace(0.0, 20.0, 201)

fig, ax = plt.subplots(figsize=(6, 4))
for g in [0.01, 0.1, 0.3, 0.45]:
    flux = [
        dust_emission(
            w10m=np.array([[w]]),
            u_t=u_t,
            gwet=np.array([[g]]),
            oro=np.array([[LAND]]),
            frlake=np.array([[0.0]]),
            S=np.array([[S]]),
            s_p=SP_DEFAULT,
        ).sum()
        for w in winds
    ]
    ax.plot(winds, flux, label=f"gwet = {g}")
ax.set_xlabel("10-m wind speed (m s$^{-1}$)")
ax.set_ylabel("Total dust flux (kg m$^{-2}$ s$^{-1}$)")
ax.set_title("GOCART dust flux")
ax.legend()
plt.tight_layout()